# HydraY NNUE: Chess324 A/B

Runtime → **GPU (T4)**, then run the cells in order. About 4h in eight stages.
**Run this notebook twice**, once per arm: the only line that changes is `ARM`
in the configuration cell.

### The question
Does Chess324 data make the net stronger? Both arms train the champion
architecture (deep16, 1024 → 16 → 1) with the champion recipe: v7, 160
superbatches, the four slices in the same order, same trainer. In every slice
**15% of the records are replaced**:

| `ARM` | the 15% comes from |
|---|---|
| `'std'` | fresh standard games (`v8_std`, labeled by the champion) |
| `'c324'` | Chess324 games (`v8_c324`, same labeler, same endgame seeding) |

The standard arm is the control: it cancels the newer labeler, so the two nets
differ only in **what** fills the replaced slots. `datamix` uses the same seed in
both arms, so the same v7 records sit in the same places.

### The verdict
A head-to-head SPRT on the laptop at fixed nodes, `c324` as new and `std` as
base, bounds [0, 5]. Chess324 joins v8 **only on H1** (ELO_ROADMAP 4.4).

### Check after the first stage
At superbatch 20 the running loss should be close to the other arm's. A much
higher or flat loss means a broken input, not a data effect: stop there.

In [ ]:
# --- helper: a failed command stops the notebook, with its output visible ---
import subprocess, os, sys, json

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')

In [ ]:
# --- configuration + Drive ---
ARM = 'c324'   # 'std' or 'c324': the ONLY line that differs between the two runs
assert ARM in ('std', 'c324')

from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert len(hits) == 1, f'{name}: {len(hits)} matches on Drive, expected 1'
    return hits[0]

PARTS = {i: find(f'hydray_v7_part{i}.bin.zst') for i in (1, 2, 3, 4)}
REPL  = {i: find(f'hydray_ab_{ARM}_part{i}.bin.zst') for i in (1, 2, 3, 4)}
# Exact record counts: datamix rejects any stream that does not match, which
# is how a decompression cut short gets caught.
V7_RECORDS   = {1: 741_965_824, 2: 741_965_824, 3: 741_965_824, 4: 742_061_977}
REPL_RECORDS = {1: 111_294_873, 2: 111_294_873, 3: 111_294_873, 4: 111_309_296}
for i in (1, 2, 3, 4):
    print(f'slice {i}: v7 {os.path.getsize(PARTS[i])/2**30:5.2f} GiB, '
          f'{ARM} {os.path.getsize(REPL[i])/2**30:4.2f} GiB compressed')

NET_ID   = f'hydray-ab-{ARM}-160sb'
TOTAL_SB = 160
STAGE    = 20
ORDER    = [1, 2, 3, 4, 1, 2, 3, 4]   # every slice twice, as for the champion
TRAINER  = '/content/th/nnue/trainer'
DATAMIX  = f'{TRAINER}/target/release/datamix'
assert len(ORDER) * STAGE == TOTAL_SB

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone, check the architecture, build datamix ---
# The checks are not ceremonial: a past run trained the wrong architecture for
# an hour because the clone was wrong and a checkpoint name says nothing.
BRANCH = 'dev'
sh('rm -rf /content/th')
sh(f'git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY /content/th')

tr = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr
assert 'const L1_SIZE: usize = 16;' in tr
assert 'const OUTPUT_BUCKETS: usize = 8;' in tr
sd = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
assert 'const INPUT_BUCKETS: usize = 4;' in sd
print(f'branch {BRANCH}: 1024 -> 16 -> 1, 4 king buckets, 8 output buckets')

sh(f'cd {TRAINER} && $HOME/.cargo/bin/cargo build -r --bin datamix')
sh('apt-get -qq install -y zstd >/dev/null')
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'{free_gb:.1f} GiB free; peak ~28 GiB (mixed slice + replacement part)')
assert free_gb > 45, 'not enough disk'

In [ ]:
# --- stage helpers (ALWAYS run this cell, also when resuming) ---

def load_slice(n):
    """/content/data.bin = v7 slice n with 15% of its records replaced."""
    for f in ('/content/data.bin', '/content/repl.bin'):
        if os.path.exists(f):
            os.remove(f)                     # free the space first
    sh(f'zstd -d -T0 --long=27 -c "{REPL[n]}" > /content/repl.bin')
    # v7 streams straight into datamix, so only the mixed slice hits the disk.
    sh(f'set -o pipefail; zstd -d -T0 --long=27 -c "{PARTS[n]}" | '
       f'{DATAMIX} /content/data.bin - {V7_RECORDS[n]} /content/repl.bin {REPL_RECORDS[n]} {n}')
    os.remove('/content/repl.bin')
    got = os.path.getsize('/content/data.bin')
    assert got == V7_RECORDS[n] * 32, f'slice {n}: {got} bytes'
    print(f'slice {n}: {V7_RECORDS[n]/1e6:.1f}M records, {REPL_RECORDS[n]/1e6:.1f}M from {ARM}', flush=True)

def stage_cmd(end, start, resume_from):
    """STAGE_END must sit right before `cargo`: `STAGE_END=40 cd dir && cargo`
    gives the variable to `cd` only, the trainer runs to TOTAL_SB and saves
    nothing. It cost a whole run once, hence the explicit `env`."""
    args = f'/content/data.bin {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer_deep --features cuda -- {args}')

def save_to_drive(end):
    """Copy the checkpoint to Drive and check it arrived: the Drive mount
    writes through a cache, so an unfinished upload would look fine."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'/content/drive/MyDrive/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint missing locally: {ck}'
    sh(f'rm -rf {dst} && cp -r {ck} /content/drive/MyDrive/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f))
                         for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst) and size(dst) == size(ck), f'incomplete copy on Drive: {dst}'
    print(f'superbatch {end} on Drive ({size(dst)/2**20:.0f} MiB, verified)', flush=True)

def run_stages(done=0):
    """Runs the stages from `done` on; done=0 starts from scratch."""
    assert done % STAGE == 0, f'{done} is not a stage boundary'
    prev = done if done else None
    for k in range(done // STAGE, len(ORDER)):
        end, start, sl = (k+1)*STAGE, k*STAGE + 1, ORDER[k]
        print(f'\n===== stage {k+1}/{len(ORDER)}: superbatches {start}-{end}, slice {sl} =====', flush=True)
        load_slice(sl)
        sh(stage_cmd(end, start, prev))
        save_to_drive(end)
        prev = end

In [ ]:
# --- training: eight stages, slices 1-2-3-4-1-2-3-4 ---
# Do NOT run this cell when resuming: use the last cell instead.
run_stages(done=0)

In [ ]:
# --- final check, sanity evals, copy to Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# 6,425,632 bytes of payload rounded up to 64; anything else is another net.
assert 6425632 <= sz < 6425632 + 64, f'size {sz}: not the 1024 -> 16 -> 1 net'
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity_deep -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print(f'\nOn Drive: {NET_ID}-{TOTAL_SB}/quantised.bin')
print('Sanity evals catch a disaster (broken mirror, absurd values), they do not')
print('predict Elo. The verdict is the SPRT.')

In [ ]:
# --- RESUME (only if the session died mid-run) ---
#   1. run the cells from "helper" through "stage helpers";
#   2. do NOT run the training cell;
#   3. set RESUME = True and DONE = the last superbatch saved on Drive.
# With RESUME = False this cell does nothing, so "Run all" stays safe.
RESUME = False
DONE   = 20

if not RESUME:
    print('resume disabled (RESUME = False)')
else:
    ck = f'/content/drive/MyDrive/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint not on Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r {ck} {TRAINER}/checkpoints/')
    print(f'resuming at superbatch {DONE} (next slice: {ORDER[DONE//STAGE]})\n')
    run_stages(done=DONE)

## After both runs

Download the two `quantised.bin` files from Drive
(`hydray-ab-std-160sb-160/` and `hydray-ab-c324-160sb-160/`) to
`nnue/net/ab_std.nnue` and `nnue/net/ab_c324.nnue` on the laptop, then run
`tuning/run_ab_c324.sh`. Same binary on both sides, the nets differ only through
`EvalFile`, fixed nodes so the running datagen cannot bias the result.

- **H1**: Chess324 data is worth at least what it costs; it joins the v8 mix.
- **H0**: no evidence of a gain of 5 Elo or more. That is not proof of harm,
  but the rule set before the test is that Chess324 stays out.